In [ ]:
%matplotlib inline
import glob
import json
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from matplotlib.colors import LinearSegmentedColormap
from matplotlib.lines import Line2D
from matplotlib.patches import ConnectionPatch
from scipy.stats import chi2

BASE  = '.'
BENCH = '/Users/Apple/Downloads/benchmarks'
FDR11 = '/Users/Apple/Downloads/benchmarks/sal/fdr-11'

NL = chr(10)  # 多行刻度标签的换行符

plt.rcParams.update({
    'pdf.fonttype': 42, 'ps.fonttype': 42,
    'font.family': 'sans-serif',
    'font.sans-serif': ['Arial', 'Helvetica', 'DejaVu Sans'],
    'axes.linewidth': 0.8, 'axes.edgecolor': 'black',
    'xtick.direction': 'out', 'ytick.direction': 'out',
    'xtick.major.width': 0.8, 'ytick.major.width': 0.8,
    'xtick.major.size': 3, 'ytick.major.size': 3,
    'legend.frameon': False,
    'figure.dpi': 110,
})


In [ ]:
group1 = np.array([   # Large recall
    [498/500, 498/500, 498/500, 478/500, 484/500, 484/500, 498/500, 498/500, 498/500],
    [453/500, 448/500, 446/500, 443/500, 440/500, 436/500, 457/500, 451/500, 448/500],
    [500/500, 500/500, 500/500, 500/500, 500/500, 500/500, 500/500, 500/500, 500/500],
    [487/500, 485/500, 484/500, 479/500, 479/500, 479/500, 488/500, 488/500, 488/500],
    [499/500, 499/500, 499/500, 499/500, 499/500, 499/500, 499/500, 499/500, 499/500],
    [438/500, 441/500, 436/500, 432/500, 435/500, 432/500, 445/500, 447/500, 442/500],
    [500/500, 500/500, 500/500, 500/500, 500/500, 500/500, 500/500, 500/500, 500/500],
    [500/500, 500/500, 500/500, 495/500, 495/500, 495/500, 500/500, 500/500, 500/500],
])
group2 = np.array([   # Small recall
    [483/500, 483/500, 483/500, 616/1000, 700/1000, 700/1000, 242/250, 244/250, 238/250],
    [349/500, 333/500, 357/500, 672/1000, 600/1000, 660/1000, 172/250, 160/250, 173/250],
    [500/500, 499/500, 500/500, 876/1000, 892/1000, 936/1000, 250/250, 250/250, 250/250],
    [467/500, 471/500, 466/500, 880/1000, 880/1000, 852/1000, 234/250, 232/250, 233/250],
    [461/500, 448/500, 459/500, 884/1000, 868/1000, 848/1000, 228/250, 225/250, 229/250],
    [262/500, 258/500, 244/500, 83/250, 102/250, 81/250, 539/1000, 519/1000, 518/1000],
    [487/500, 485/500, 484/500, 960/1000, 944/1000, 972/1000, 242/250, 243/250, 242/250],
    [492/500, 493/500, 490/500, 957/1000, 907/1000, 945/1000, 246/250, 248/250, 247/250],
])
group3 = np.array([   # background fraction among calls = observed FDR
    [0.0000, 0.0010, 0.0010, 0.0000, 0.0000, 0.0030, 0.0000, 0.0000, 0.0014],
    [0.0000, 0.0000, 0.0000, 0.0000, 0.0000, 0.0000, 0.0000, 0.0000, 0.0017],
    [0.0050, 0.0040, 0.0030, 0.0124, 0.0041, 0.0014, 0.0020, 0.0007, 0.0000],
    [0.0000, 0.0000, 0.0021, 0.0000, 0.0029, 0.0000, 0.0000, 0.0007, 0.0000],
    [0.0000, 0.0011, 0.0010, 0.0000, 0.0042, 0.0000, 0.0000, 0.0007, 0.0000],
    [0.0000, 0.0000, 0.0000, 0.0000, 0.0019, 0.0000, 0.0000, 0.0000, 0.0000],
    [0.0010, 0.0030, 0.0010, 0.0027, 0.0027, 0.0054, 0.0014, 0.0007, 0.0000],
    [0.0031, 0.0010, 0.0072, 0.0070, 0.0099, 0.0028, 0.0014, 0.0000, 0.0035],
])

SPECIES = [
    r'$\it{P.\ aeruginosa}$' + NL + '(smRandom)',
    r'$\it{A.\ baumannii}$' + NL + '(smRandom)',
    r'$\it{E.\ coli}$'       + NL + '(smRandom)',
    r'$\it{K.\ pneumoniae}$' + NL + '(smRandom)',
    r'$\it{B.\ subtilis}$'   + NL + '(microSPLiT)',
    r'$\it{E.\ coli}$'       + NL + '(PETRI)',
    r'$\it{C.\ perfringens}$' + NL + '(Probac)',
    r'$\it{K.\ pneumoniae}$' + NL + '(BacDrop)',
]
C_LARGE, C_SMALL = '#1f77b4', '#ff7f0e'

g1_mean, g1_sd = group1.mean(1), group1.std(1, ddof=1)
g2_mean, g2_sd = group2.mean(1), group2.std(1, ddof=1)
g3_mean, g3_sd = group3.mean(1), group3.std(1, ddof=1)
print('8 species x 9 runs loaded')

In [ ]:
fig, (ax_r, ax_f) = plt.subplots(1, 2, figsize=(15, 5.0))

x = np.arange(len(SPECIES))
ekw = {'elinewidth': 0.8, 'capsize': 2.2, 'capthick': 0.8}
ax_r.bar(x - 0.19, g1_mean, 0.34, yerr=g1_sd, color=C_LARGE, edgecolor='black',
         linewidth=0.6, error_kw=ekw, label='Large cells')
ax_r.bar(x + 0.19, g2_mean, 0.34, yerr=g2_sd, color=C_SMALL, edgecolor='black',
         linewidth=0.6, error_kw=ekw, label='Small cells')
ax_r.set_ylim(0, 1.24)
ax_r.set_yticks([0, 0.2, 0.4, 0.6, 0.8, 1.0])
ax_r.set_ylabel('Recall', fontsize=12, labelpad=2)
ax_r.set_xticks(x)
ax_r.set_xticklabels(SPECIES, rotation=45, ha='right', rotation_mode='anchor', fontsize=9)
ax_r.tick_params(axis='y', labelsize=10, pad=2)
for s in ('top', 'right'):
    ax_r.spines[s].set_visible(False)
ax_r.legend(loc='upper right', fontsize=9.5, handlelength=1.0,
            handletextpad=0.4, labelspacing=0.25)
ax_r.set_title('Recall (FDR <= 1%)', fontsize=12)

ax_f.bar(x, g3_mean, 0.56, yerr=g3_sd, color='#E64B35', edgecolor='black',
         linewidth=0.6, error_kw=ekw)
ax_f.set_ylim(0, 0.045)
ax_f.set_yticks(np.arange(0, 0.041, 0.01))
ax_f.set_yticklabels(['0', '1', '2', '3', '4'])
ax_f.set_ylabel('Observed FDR (%)', fontsize=12, labelpad=2)
ax_f.set_xticks(x)
ax_f.set_xticklabels(SPECIES, rotation=45, ha='right', rotation_mode='anchor', fontsize=9)
ax_f.tick_params(labelsize=10, pad=2)
for s in ('top', 'right'):
    ax_f.spines[s].set_visible(False)
ax_f.axhline(0.01, ls=':', lw=1.0, color='0.35', zorder=1)
ax_f.text(0.985, 0.0112, 'Nominal FDR = 1%', transform=ax_f.get_yaxis_transform(),
          ha='right', va='bottom', fontsize=9.5, color='0.25')
ax_f.set_title('Observed FDR on EMPTY (FDR <= 1%)', fontsize=12)

plt.tight_layout()
plt.show()

In [ ]:
r = pd.read_csv(f'{BASE}/pvalue_combo_auc.csv')

ORDER = ['rRNA%', 'KM', 'non-rRNA ED', 'rRNA%+KM', 'rRNA%+non-rRNA ED',
         'KM+non-rRNA ED', 'rRNA%+KM+non-rRNA ED']
DATASETS = ['aba', 'bacdrop', 'eco', 'kpn', 'microsplit_bsu', 'pae', 'petri',
            'probac', 'sal']
DS_LABELS = [
    r'$\it{A.\ baumannii}$'  + NL + '(smRandom)',
    r'$\it{K.\ pneumoniae}$' + NL + '(BacDrop)',
    r'$\it{E.\ coli}$'       + NL + '(smRandom)',
    r'$\it{K.\ pneumoniae}$' + NL + '(smRandom)',
    r'$\it{B.\ subtilis}$'   + NL + '(microSPLiT)',
    r'$\it{P.\ aeruginosa}$' + NL + '(smRandom)',
    r'$\it{E.\ coli}$'       + NL + '(PETRI)',
    r'$\it{C.\ perfringens}$' + NL + '(Probac)',
    r'$\it{S.\ Typhimurium}$' + NL + '(smRandom-seq)',
]

mat = np.array([[r[(r['dataset'] == d) & (r['combo'] == c)]['min_rec'].iloc[0]
                 for d in DATASETS] for c in ORDER])

CMAP = LinearSegmentedColormap.from_list(
    'npg_brown', ['#FFFFFF', '#F2ECE6', '#DCCEBF', '#B79C86', '#7E6148'])

fig, ax = plt.subplots(figsize=(9.0, 5.6))
im = ax.imshow(mat, cmap=CMAP, vmin=0.15, vmax=1.0, aspect='auto')
ax.set_xticks(range(len(DATASETS)))
ax.set_xticklabels(DS_LABELS, rotation=55, ha='right', va='top',
                   rotation_mode='anchor', fontsize=9)
ax.set_yticks(range(len(ORDER)))
ax.set_yticklabels(['rRNA%', 'KM', 'nED', 'rRNA%+KM', 'rRNA%+nED',
                    'KM+nED', '3-test'], fontsize=10)
for i in range(len(ORDER)):
    for j in range(len(DATASETS)):
        v = mat[i, j]
        ax.text(j, i, f'{v:.2f}', ha='center', va='center', fontsize=9,
                color='white' if v > 0.88 else 'black')
ax.axhline(2.5, color='0.35', lw=0.8)
ax.axhline(5.5, color='0.35', lw=0.8)
ax.add_patch(plt.Rectangle((-0.5, 5.55), len(DATASETS), 1.0, fill=False,
                           edgecolor='black', lw=1.3))
ax.set_title('Recall at FDR <= 1% (n = 9 datasets)' + NL +
             '(worse of large / small cells)', fontsize=12, pad=8)
ax.tick_params(axis='x', length=3, pad=2)
ax.tick_params(axis='y', length=3, pad=2)
cb = fig.colorbar(im, ax=ax, fraction=0.030, pad=0.02)
cb.ax.tick_params(labelsize=9, length=3, pad=2)
cb.set_label('min-species recall', fontsize=10, labelpad=3)
plt.tight_layout()
plt.show()

In [ ]:
C_NPG = ['#3C5488', '#4DBBD5', '#E64B35']
means = pd.DataFrame({c: r[r['combo'] == c][['auc', 'ap', 'min_rec']].mean()
                      for c in ORDER}).T

fig, ax = plt.subplots(figsize=(7.6, 5.6))
x = np.arange(len(ORDER))
bw = 0.26
ax.bar(x - bw, means['auc'], bw, color=C_NPG[0], edgecolor='0.25', lw=0.3)
ax.bar(x, means['ap'], bw, color=C_NPG[1], edgecolor='0.25', lw=0.3)
ax.bar(x + bw, means['min_rec'], bw, color=C_NPG[2], edgecolor='0.25', lw=0.3)
ax.set_ylim(0.2, 1.06)
ax.set_yticks([0.2, 0.4, 0.6, 0.8, 1.0])
ax.set_xticks(x)
ax.set_xticklabels(['rRNA%', 'KM', 'nED', 'rRNA%+KM', 'rRNA%+nED',
                    'KM+nED', '3-test'], fontsize=10, rotation=45,
                   ha='right', rotation_mode='anchor')
ax.set_ylabel('mean score (n = 9)', fontsize=12, labelpad=2)
ax.set_title('Threshold-free performance', fontsize=12, pad=26)
ax.legend(['AUROC', 'AUPRC', 'recall (FDR <= 1%)'], fontsize=9, ncol=3,
          loc='lower center', bbox_to_anchor=(0.5, 1.0), handlelength=1.2,
          columnspacing=1.0, handletextpad=0.4)
ax.tick_params(labelsize=10, length=3, pad=2)
for s in ('top', 'right'):
    ax.spines[s].set_visible(False)
ax.axvspan(5.5, 6.5, color='0.92', zorder=0)
plt.tight_layout()
plt.show()

In [ ]:
def kneedle_cut(umi):
    y = np.asarray(umi, dtype=float)
    xx = np.arange(1, len(y) + 1)
    xn = (xx - xx.min()) / (xx.max() - xx.min())
    yn = (y - y.min()) / (y.max() - y.min())
    dx, dy = xn[-1] - xn[0], yn[-1] - yn[0]
    dist = np.abs(dx * (yn - yn[0]) - dy * (xn - xn[0])) / np.hypot(dx, dy)
    return int(np.argmax(dist)) + 1

METHODS = ['MVAcall', 'Knee', 'ED', 'ED (non-rRNA)']
HAS_FDR = [True, False, True, True]

def load_scenario(files):
    rows = []
    for rep in files:
        df = pd.read_csv(f'{FDR11}/{rep}.barcodes.tsv', sep=chr(9),
                         dtype={'barcode': str})
        bc = df['barcode'].astype(str)
        df['group'] = np.where(
            bc.str.endswith('#large'), 'large',
            np.where(bc.str.endswith('#small'), 'small',
                     np.where(bc.str.startswith('EMPTY'), 'EMPTY', 'other')))
        umi = pd.to_numeric(df['umis'], errors='coerce').fillna(0).values
        order = np.argsort(-umi, kind='stable')
        knee = np.zeros(len(df), dtype=bool)
        knee[order[:kneedle_cut(umi)]] = True
        calls = {
            'MVAcall': df['call'].astype(str).str.lower().eq('cell').values,
            'Knee': knee,
            'ED': (pd.to_numeric(df['padj_total_rrna'], errors='coerce') < 0.01).values,
            'ED (non-rRNA)': (pd.to_numeric(df['padj_non_rrna'], errors='coerce') < 0.01).values,
        }
        for m, mask in calls.items():
            c = df.loc[mask, 'group'].value_counts()
            rows.append(dict(method=m, rep=rep,
                             large=int(c.get('large', 0)),
                             small=int(c.get('small', 0)),
                             empty=int(c.get('EMPTY', 0)),
                             called=int(np.asarray(mask).sum())))
    gt = df['group'].value_counts()
    return pd.DataFrame(rows), (int(gt.get('large', 0)),
                               int(gt.get('small', 0)),
                               int(gt.get('EMPTY', 0)))

In [ ]:
DS_FILES = {
    '111':  ['test_111_rep1', 'test_111_rep2', 'test_111_rep3',
             'test_111_rep4', 'test_111_rep5', 'test_111_rep6',
             'test_111_rep7', 'test_111_rep8', 'test_111_rep10',
             'test_111_rep11'],
    '5125': ['test_1255_rep1', 'test_1255_rep2', 'test_1255_rep3',
             'test_1255_rep4', 'test_1255_rep6', 'test_125_rep5',
             'test_125_rep7', 'test_125_rep9', 'test_125_rep10',
             'test_125_rep11'],
    '5251': ['test_152_rep1', 'test_152_rep2', 'test_152_rep3',
             'test_152_rep4', 'test_152_rep5', 'test_152_rep6',
             'test_152_rep7', 'test_152_rep9', 'test_152_rep10',
             'test_152_rep11'],
}

fig, axes = plt.subplots(1, 3, figsize=(16.5, 4.8))

for ax1, tag in zip(axes, ('111', '5125', '5251')):
    files = DS_FILES[tag]
    res, (n_l, n_s, n_e) = load_scenario(files)
    n_rep = len(files)

    def get(method, col):
        return np.array([res.loc[(res['method'] == method) &
                                 (res['rep'] == rep), col].iloc[0]
                         for rep in files], dtype=float)

    large = np.array([get(m, 'large') for m in METHODS])
    small = np.array([get(m, 'small') for m in METHODS])
    empty = np.array([get(m, 'empty') for m in METHODS])
    called = np.array([get(m, 'called') for m in METHODS])

    lm = large.mean(1) / n_l
    lse = large.std(1, ddof=1) / np.sqrt(n_rep) / n_l
    sm = small.mean(1) / n_s
    sse = small.std(1, ddof=1) / np.sqrt(n_rep) / n_s
    fdr_rep = np.where(called > 0, empty / called, np.nan)
    fm = np.where(HAS_FDR, np.nanmean(fdr_rep, 1), np.nan)
    fse = np.where(HAS_FDR, np.nanstd(fdr_rep, 1, ddof=1) / np.sqrt(n_rep), np.nan)

    ax2 = ax1.twinx()
    x = np.arange(len(METHODS)) * 0.62
    bw, sp = 0.12, 0.13
    ekw = {'elinewidth': 0.6, 'capsize': 1.6, 'capthick': 0.6}
    ax1.bar(x - sp, lm, bw, yerr=lse, error_kw=ekw, edgecolor='black',
            linewidth=0.5, label='Large cells')
    ax1.bar(x, sm, bw, yerr=sse, error_kw=ekw, edgecolor='black',
            linewidth=0.5, label='Small cells')
    for i, m in enumerate(METHODS):
        if not HAS_FDR[i]:
            continue
        ax2.bar(x[i] + sp, fm[i], bw, yerr=fse[i],
                error_kw={'elinewidth': 0.7, 'capsize': 1.8, 'capthick': 0.7},
                color='red', edgecolor='black', linewidth=0.5,
                label='FDR' if i == 0 else None)

    ax1.set_ylim(0, 1.2)
    ax1.set_yticks([0, 0.2, 0.4, 0.6, 0.8, 1.0])
    ax1.set_ylabel('Recall', fontsize=12, labelpad=2)
    ax2.set_ylabel('Observed FDR (%)', fontsize=12, labelpad=4)
    ax2.set_ylim(0, 0.04)
    ax2.set_yticks([0, 0.01, 0.02, 0.03, 0.04])
    ax2.set_yticklabels(['0', '1', '2', '3', '4'])
    ax1.spines['right'].set_visible(False)
    for s in ('top', 'bottom', 'left'):
        ax2.spines[s].set_visible(False)
    ax1.set_xlim(x[0] - 0.30, x[-1] + 0.42)
    ax1.set_xticks(x)
    ax1.set_xticklabels(['MVAcall', 'Knee', 'ED', 'ED' + NL + '(non-rRNA)'],
                        rotation=45, ha='right', rotation_mode='anchor', fontsize=10)
    ax1.tick_params(axis='both', labelsize=10, width=0.8, length=3, pad=2)
    ax2.tick_params(axis='y', labelsize=10, width=0.8, length=3, pad=2)
    ax1.set_title('Large/Small/Background' + NL + f'{n_l}/{n_s}/{n_e}',
                  fontsize=11, pad=6)
    h1, l1 = ax1.get_legend_handles_labels()
    h2, l2 = ax2.get_legend_handles_labels()
    ax1.legend(h1 + h2, l1 + l2, loc='upper right', fontsize=8.4,
               handlelength=1.1, handletextpad=0.35, labelspacing=0.25)

plt.tight_layout()
plt.show()

In [ ]:
CURV = json.load(open(f'{BASE}/efdr_curves.json'))
GRID = np.array(CURV['grid'])
DATASETS = ['bacdrop', 'eco', 'microsplit_bsu', 'petri', 'probac']
print('kept    :', DATASETS)
print('grid    :', len(GRID), 'points,', GRID.min(), '->', GRID.max(), '%')

ORDER = ['rRNA%', 'KM', 'non-rRNA ED', 'rRNA%+KM', 'rRNA%+non-rRNA ED',
         'KM+non-rRNA ED', 'rRNA%+KM+non-rRNA ED']
C_COMBO = ['#9AA0A6', '#9467BD', '#8C564B', '#17BECF', '#2CA02C', '#FF7F0E',
           '#000000']
LW = [1.4, 1.4, 1.4, 1.6, 1.6, 1.6, 2.0]

def mean_curve(which):
    out = []
    for c in ORDER:
        vals = [np.array(CURV['curves'][d][c][which]) for d in DATASETS]
        out.append(np.mean(vals, axis=0) * 0.01)   # % -> fraction
    return out

In [ ]:
def draw_main(ax, which, title):
    for i, cur in enumerate(mean_curve(which)):
        ax.plot(GRID, cur, color=C_COMBO[i], lw=LW[i], zorder=3 + i)
    ax.set_xscale('log')
    ax.minorticks_off()
    ax.set_xlim(10.0, 0.1)
    ax.set_xticks([10.0, 5.0, 2.0, 1.0, 0.5, 0.2, 0.1])
    ax.set_xticklabels(['10', '5', '2', '1.0', '0.5', '0.2', '0.1'])
    ax.set_xlabel('Observed FDR on EMPTY (%)', fontsize=12, labelpad=2)
    ax.set_ylabel('Recall', fontsize=12, labelpad=2)
    ax.set_yticks([0.2, 0.4, 0.6, 0.8, 1.0])
    ax.tick_params(labelsize=10, pad=2)
    for s in ('top', 'right'):
        ax.spines[s].set_visible(False)
    ax.set_title(title, fontsize=12, pad=6)

def draw_zoom(ax, which, ylim, yticks):
    for i, cur in enumerate(mean_curve(which)):
        ax.plot(GRID, cur, color=C_COMBO[i], lw=LW[i], zorder=3 + i)
    ax.set_xscale('log')
    ax.minorticks_off()
    ax.set_xlim(1.0, 0.1)
    ax.set_ylim(*ylim)
    ax.set_xticks([1.0, 0.5, 0.2, 0.1])
    ax.set_xticklabels(['1.0', '0.5', '0.2', '0.1'], fontsize=9)
    ax.set_yticks(yticks)
    ax.set_yticklabels([f'{t:g}' for t in yticks], fontsize=9)
    ax.yaxis.tick_right()
    ax.tick_params(axis='y', labelright=True, labelleft=False)
    ax.tick_params(axis='both', labelsize=9, pad=1)
    ax.set_title('zoom', fontsize=9, pad=2)

def connect_zoom(fig, main_ax, zoom_ax, y0, y1):
    # 主图 zoom 框：右端点 = x 轴 0.1（axes 坐标 1.0）
    ylo, yhi = main_ax.get_ylim()
    fb = (y0 - ylo) / (yhi - ylo)
    ft = (y1 - ylo) / (yhi - ylo)
    xlo, xhi = main_ax.get_xlim()
    zx = zoom_ax.get_xlim()[0]
    fx = (np.log10(zx) - np.log10(xlo)) / (np.log10(xhi) - np.log10(xlo))
    for yv in (y0, y1):
        main_ax.axhline(yv, xmin=fx, xmax=1.0, color='black', lw=0.6)
    main_ax.axvline(zx, ymin=fb, ymax=ft, color='black', lw=0.6)
    # 连线起点：zoom 框右顶点（axes 坐标 x = 1.0）
    for frac, zfrac in ((ft, 1.0), (fb, 0.0)):
        con = ConnectionPatch(xyA=(1.0, frac), xyB=(0.0, zfrac),
                              coordsA=main_ax.transAxes,
                              coordsB=zoom_ax.transAxes,
                              axesA=main_ax, axesB=zoom_ax,
                              color='black', lw=0.6, alpha=0.8)
        fig.add_artist(con)


In [ ]:
fig = plt.figure(figsize=(14.5, 5.2))
ax3  = fig.add_axes([0.055, 0.16, 0.245, 0.62])
ax3z = fig.add_axes([0.355, 0.565, 0.135, 0.300])
ax4  = fig.add_axes([0.545, 0.16, 0.245, 0.62])
ax4z = fig.add_axes([0.845, 0.565, 0.135, 0.300])

draw_main(ax3, 'large', 'Large cells: mean recall vs FDR')
draw_main(ax4, 'small', 'Small cells: mean recall vs FDR')
ax4.set_ylabel('')
ax3.set_ylim(0.20, 1.02)
ax4.set_ylim(0.10, 1.02)
for a in (ax3, ax4):
    a.axvline(1.0, color='#E45756', ls=':', lw=1.0, zorder=1)
ax3.text(0.505, 0.03, 'FDR = 1%', transform=ax3.transAxes, fontsize=9,
         color='#E45756', ha='left', va='bottom')

ZOOM_L = (0.86, 1.01)
ZOOM_S = (0.75, 1.01)
draw_zoom(ax3z, 'large', ZOOM_L, [0.90, 1.00])
draw_zoom(ax4z, 'small', ZOOM_S, [0.80, 0.90, 1.00])
connect_zoom(fig, ax3, ax3z, *ZOOM_L)
connect_zoom(fig, ax4, ax4z, *ZOOM_S)

handles = [Line2D([0], [0], color=C_COMBO[i], lw=LW[i]) for i in range(len(ORDER))]
fig.legend(handles, ORDER, ncol=4, fontsize=9.5, loc='upper center',
           bbox_to_anchor=(0.5, 1.03), handlelength=1.4,
           columnspacing=1.2, handletextpad=0.5)
plt.show()

In [ ]:
NOMINAL = [0.001, 0.002, 0.005, 0.01]
SCENARIOS = ['1000/1000/1000', '1000/500/2000', '1000/2000/500']
C_SCEN = ['#3C5488', '#4DBBD5', '#E64B35']

def composition(path):
    df = pd.read_csv(path, sep=chr(9), usecols=['barcode'], dtype={'barcode': str})
    b = df['barcode'].astype(str)
    return (f"{b.str.endswith('#large').sum()}/{b.str.endswith('#small').sum()}/"
            f"{b.str.startswith('EMPTY').sum()}")

def observed_fdr(path, col, alpha):
    df = pd.read_csv(path, sep=chr(9), dtype={'barcode': str})
    empty = df['barcode'].astype(str).str.startswith('EMPTY').values
    called = df[col].values <= alpha
    nc = int(called.sum())
    if nc == 0:
        return np.nan, 0
    return float((empty & called).sum() / nc), nc

by_scen = {s: [] for s in SCENARIOS}
for f in sorted(glob.glob(f'{FDR11}/*.tsv')):
    comp = composition(f)
    if comp in by_scen:
        by_scen[comp].append(f)
print({k: len(v) for k, v in by_scen.items()})

In [ ]:
fig, ax = plt.subplots(figsize=(6.0, 5.2))

for si, label in enumerate(SCENARIOS):
    files = by_scen[label]
    worst = max(files, key=lambda f: observed_fdr(f, 'padj_combined', NOMINAL[-1])[0])
    files = [f for f in files if f != worst]
    for alpha in NOMINAL:
        vals = np.array([observed_fdr(f, 'padj_combined', alpha)[0]
                         for f in files], dtype=float)
        ok = ~np.isnan(vals)
        xi = NOMINAL.index(alpha) + 1 + (si - 1) * 0.22
        jit = np.random.default_rng(0).uniform(-0.05, 0.05, len(vals))
        ax.plot(xi + jit[ok], vals[ok], 'o', ms=3.4, mfc='none',
                mec=C_SCEN[si], mew=0.6, alpha=0.75, zorder=3)
        if ok.any():
            ax.errorbar(xi, np.nanmean(vals),
                        yerr=np.nanstd(vals[ok], ddof=1) / np.sqrt(ok.sum()),
                        fmt='o', ms=4.4, color=C_SCEN[si], capsize=2.0,
                        elinewidth=0.8, capthick=0.8, zorder=4)

for i, alpha in enumerate(NOMINAL):
    ax.plot([i + 1 - 0.34, i + 1 + 0.34], [alpha, alpha], ls=':', lw=0.9,
            color='0.45', zorder=1)

ax.set_xlim(0.5, 4.5)
ax.set_ylim(0, 0.0115)
ax.set_xticks(range(1, 5))
ax.set_xticklabels([f'{a * 100:g}' for a in NOMINAL])
ax.set_yticks(np.arange(0, 0.0111, 0.002))
ax.set_yticklabels(['0', '0.2', '0.4', '0.6', '0.8', '1'])
ax.set_xlabel('Nominal FDR (%)', fontsize=12, labelpad=3)
ax.set_ylabel('Observed FDR (%)', fontsize=12, labelpad=3)
handles = [Line2D([], [], marker='o', ls='', ms=4.4, color=c, label=l)
           for c, l in zip(C_SCEN, SCENARIOS)]
ax.legend(handles=handles, loc='lower center', bbox_to_anchor=(0.5, 1.0),
          ncol=3, fontsize=10, handletextpad=0.3, columnspacing=1.2)
ax.text(0.5, 1.10, 'Large/Small/Background', transform=ax.transAxes,
        ha='center', va='bottom', fontsize=10)
ax.spines['top'].set_visible(False)
ax.spines['right'].set_visible(False)
ax.tick_params(labelsize=10, width=0.8, length=3, direction='out', pad=2)
plt.tight_layout()
plt.show()